In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

DASHSCOPE_API_URL = os.getenv("DASHSCOPE_API_URL")
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DEFAULT_LLM_MODEL = os.getenv("DEFAULT_LLM_MODEL")
print(DASHSCOPE_API_KEY, DASHSCOPE_API_URL, DEFAULT_LLM_MODEL)

sk-d04de33360574dd6bd8224dc1d9897a3 https://dashscope.aliyuncs.com/compatible-mode/v1 qwen-turbo


In [16]:
from langchain.agents import AgentType
from langchain_core.tools import Tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import initialize_agent
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts.chat import MessagesPlaceholder

default_config = {
    'api_key': DASHSCOPE_API_KEY,
    'base_url': DASHSCOPE_API_URL,
    'temperature': 0.1,
    "extra_body": {
        'enable_thinking': False
    }
}
tools = [
    Tool(name="Search", func=lambda q:"100块", description="这是一个价格搜索工具"),
]
llm = ChatOpenAI(model='qwen-turbo', **default_config)
parser = JsonOutputParser()
print(parser.get_format_instructions())
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="你是一个商品报价客服。"),
    HumanMessage(content="用户输入：{user_input}，你的需要返回：{{‘商品’：商品名称，'price'；商品价格}}"),
    MessagesPlaceholder("agent_scratchpad"),
])
agent = initialize_agent(tools, llm, agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION)
# 仅适用于无工具场景
# chain = prompt | llm | parser
# response = chain.invoke({
#     "response_instructions": parser.get_format_instructions(),
#     "user_input":"鼠标多少钱？"
# })
query = "鼠标多少钱？"
response = agent.invoke(query)
response

Return a JSON object.


{'input': '鼠标多少钱？', 'output': '鼠标的价格是100块。'}

In [13]:
from langchain.agents import create_openai_functions_agent
from langchain.agents import AgentExecutor
query = "鼠标多少钱？"
agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)

In [18]:
agent_executor.invoke({'user_input': query})

{'user_input': '鼠标多少钱？', 'output': '请提供您想要查询的商品名称。'}